# Phase 2: 10-Class MNIST Classification — K-Nearest Neighbors
## Pipeline: Data Loading -> Preprocessing -> Feature Extraction -> Training -> CV Tuning -> Learning Curves -> Evaluation

## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from ml_utils import train_test_split, StandardScaler
from ml_utils import classification_report, confusion_matrix
from ml_utils import compute_accuracy, k_fold_split, cross_validate
from ml_utils import plot_learning_curve, evaluate_model, show_misclassified
from sklearn.decomposition import PCA
from skimage.feature import hog
np.random.seed(42)
print("Libraries imported")

## 2. Configuration

In [ ]:
NUM_CLASSES = 10
TRAIN_SIZE, VAL_SIZE, TEST_SIZE = 0.70, 0.15, 0.15
PCA_VARIANCE = 0.95
K_FOLDS = 5
LC_FRACTIONS = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
KNN_K_GRID = [1, 3, 5, 7, 9]
DEFAULT_K = 5

## 3. Data Loading & Preprocessing

In [ ]:
X_train_raw = np.load('../../mnist.npz/x_train.npy')
y_train_raw = np.load('../../mnist.npz/y_train.npy')
X_test_raw  = np.load('../../mnist.npz/x_test.npy')
y_test_raw  = np.load('../../mnist.npz/y_test.npy')
X_all = np.concatenate([X_train_raw, X_test_raw], axis=0)
y_all = np.concatenate([y_train_raw, y_test_raw], axis=0)

min_count = min(np.bincount(y_all))
balanced_idx = []
for c in range(NUM_CLASSES):
    c_idx = np.where(y_all == c)[0]
    np.random.shuffle(c_idx)
    balanced_idx.extend(c_idx[:min_count])
balanced_idx = np.array(balanced_idx)
np.random.shuffle(balanced_idx)
X_balanced, y_balanced = X_all[balanced_idx], y_all[balanced_idx]
X_norm = X_balanced / 255.0

X_temp, X_test, y_temp, y_test = train_test_split(
    X_norm, y_balanced, test_size=TEST_SIZE/(TRAIN_SIZE+VAL_SIZE+TEST_SIZE),
    random_state=42, stratify=y_balanced)
val_adj = VAL_SIZE / (TRAIN_SIZE + VAL_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_adj, random_state=42, stratify=y_temp)
print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

### Visualise Dataset Samples
One example per class from the training set.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for c in range(NUM_CLASSES):
    r, col = divmod(c, 5)
    idx = np.where(y_train == c)[0][0]
    axes[r, col].imshow(X_train[idx], cmap='gray')
    axes[r, col].set_title(f'Digit {c}', fontweight='bold')
    axes[r, col].axis('off')
plt.suptitle('Training Set — One Sample per Class', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 4a. Flatten + Scaler + PCA Feature Extraction

In [ ]:
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat = X_val.reshape(X_val.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_flat)
X_val_sc = scaler.transform(X_val_flat)
X_test_sc = scaler.transform(X_test_flat)
pca = PCA(n_components=PCA_VARIANCE, svd_solver='full')
X_train_pca = pca.fit_transform(X_train_sc)
X_val_pca = pca.transform(X_val_sc)
X_test_pca = pca.transform(X_test_sc)
print(f"PCA: {X_train_sc.shape[1]} → {X_train_pca.shape[1]} components ({np.sum(pca.explained_variance_ratio_):.4f} var)")

### Visualise PCA

In [ ]:
cum_var = np.cumsum(pca.explained_variance_ratio_)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(cum_var, linewidth=2)
axes[0].axhline(PCA_VARIANCE, ls='--', color='red', label=f'{PCA_VARIANCE:.0%} threshold')
axes[0].set_xlabel('# Components'); axes[0].set_ylabel('Cumulative Variance')
axes[0].set_title('PCA Explained Variance'); axes[0].legend(); axes[0].grid(alpha=.3)

recon = pca.inverse_transform(X_train_pca[0:1])
recon = scaler.inverse_transform(recon).reshape(28, 28)
axes[1].imshow(recon, cmap='gray')
axes[1].set_title(f'Reconstructed ({X_train_pca.shape[1]} PCs)'); axes[1].axis('off')
plt.suptitle('PCA Feature Extraction', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 4b. HOG Feature Extraction

**Histogram of Oriented Gradients** captures edge and gradient structure.
Applied to the 2D images (before flattening), then standardised.

In [ ]:
def extract_hog(images):
    feats = []
    for img in images:
        f = hog(img, orientations=9, pixels_per_cell=(4,4),
                cells_per_block=(2,2), block_norm='L2-Hys')
        feats.append(f)
    return np.array(feats)

X_train_hog = extract_hog(X_train)
X_val_hog   = extract_hog(X_val)
X_test_hog  = extract_hog(X_test)

scaler_hog = StandardScaler()
X_train_hog_sc = scaler_hog.fit_transform(X_train_hog)
X_val_hog_sc   = scaler_hog.transform(X_val_hog)
X_test_hog_sc  = scaler_hog.transform(X_test_hog)

print(f"HOG feature dim: {X_train_hog_sc.shape[1]}")
print("HOG features ready")

### Visualise HOG Features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(X_train[0], cmap='gray')
axes[0].set_title('Original'); axes[0].axis('off')
_, hog_img = hog(X_train[0], orientations=9, pixels_per_cell=(4,4),
                 cells_per_block=(2,2), block_norm='L2-Hys', visualize=True)
axes[1].imshow(hog_img, cmap='hot')
axes[1].set_title('HOG Visualisation'); axes[1].axis('off')
plt.suptitle('HOG Feature Extraction', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 6. KNN Model

Uses **Euclidean distance** and **majority voting**.

In [ ]:
class KNearestNeighbors:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        self.X_train, self.y_train = X, y
        return self

    def _predict_single(self, x):
        dists = np.sqrt(np.sum((self.X_train - x)**2, axis=1))
        k_labels = self.y_train[np.argsort(dists)[:self.k]]
        return np.argmax(np.bincount(k_labels, minlength=10))

    def predict(self, X):
        return np.array([self._predict_single(X[i]) for i in tqdm(range(len(X)), desc='KNN')])

## 7. Baseline Training (PCA Features)

In [ ]:
print("BASELINE KNN (k=5)")
knn_base = KNearestNeighbors(k=DEFAULT_K)
knn_base.fit(X_train_pca, y_train)
base_acc = evaluate_model(knn_base, X_test_pca, y_test, "KNN Baseline (k=5)")

In [ ]:
# Note: showing original images for misclassified PCA predictions
preds_pca = knn_base.predict(X_test_pca)
show_misclassified(y_test, preds_pca, X_test,
                   title='Misclassified — PCA Features')

### KNN — Flatten Features

In [ ]:
print("\n" + "="*70)
print("KNN — Flatten Features")
print("="*70)
model_flatten = KNearestNeighbors(k=DEFAULT_K)
model_flatten.fit(X_train_sc, y_train)
evaluate_model(model_flatten, X_test_sc, y_test, "KNN (Flatten)")

In [ ]:
# Note: showing original images for misclassified Flatten predictions
preds_flatten = model_flatten.predict(X_test_sc)
show_misclassified(y_test, preds_flatten, X_test,
                   title='Misclassified — Flatten Features')

### KNN — HOG Features

In [ ]:
print("\n" + "="*70)
print("KNN — HOG Features")
print("="*70)
model_hog = KNearestNeighbors(k=DEFAULT_K)
model_hog.fit(X_train_hog_sc, y_train)
evaluate_model(model_hog, X_test_hog_sc, y_test, "KNN (HOG)")

In [ ]:
# Note: showing original images for misclassified HOG predictions
preds_hog = model_hog.predict(X_test_hog_sc)
show_misclassified(y_test, preds_hog, X_test,
                   title='Misclassified — HOG Features')

## 8. Hyperparameter Tuning (K-Fold CV)

In [ ]:
best_acc, best_k = 0, DEFAULT_K
cv_res = []
for kv in KNN_K_GRID:
    print(f"\nK={kv}")
    a = cross_validate(KNearestNeighbors, {'k': kv}, X_train_pca, y_train, k=K_FOLDS)
    cv_res.append((kv, a))
    if a > best_acc: best_acc, best_k = a, kv
print(f"\nBest K={best_k} (CV={best_acc:.4f})")

ks, accs = zip(*cv_res)
plt.figure(figsize=(7,4))
plt.plot(ks, accs, 'o-', color='#3498db', lw=2, ms=8)
plt.xlabel('K'); plt.ylabel('CV Accuracy')
plt.title('KNN Tuning', fontweight='bold'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
knn_tuned = KNearestNeighbors(k=best_k)
knn_tuned.fit(X_train_pca, y_train)
tuned_acc = evaluate_model(knn_tuned, X_test_pca, y_test, f"KNN Tuned (k={best_k})")

In [ ]:
# Note: showing original images for misclassified PCA predictions
preds_pca = knn_tuned.predict(X_test_pca)
show_misclassified(y_test, preds_pca, X_test,
                   title='Misclassified — PCA Features')

## 9. Learning Curve

In [ ]:
plot_learning_curve(KNearestNeighbors, {'k': best_k}, X_train_pca, y_train,
                    X_val_pca, y_val, LC_FRACTIONS, f"Learning Curve — KNN (k={best_k})")

## 10. Summary

**Analysis**: KNN with small k → high variance (train≈100%, val lower).
Increasing k reduces variance but increases bias.